# 01 Data understanding

**Purpose:** get to know every file before changing anything. This notebook only reads and describes the data. Cleaning happens in notebook 02, and labels are built in notebook 03.

**Inputs:** the raw data files in `data/raw/`.

**Outputs:** none. The findings here shape the decisions in the later notebooks, and they are summarised at the end.

**Questions this notebook answers**
1. What is in each file, and how do the files connect?
2. What does the delivery network look like (outlets, vehicles, depots)?
3. How do orders and route legs relate, and what period do train and test cover?
4. How often are deliveries late, and what seems to drive it?
5. How does weekly demand move, especially around festivals?
6. How tight is the Task 2B peak day?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora.io import load_raw
from xora.paths import DATA_RAW
from xora.timeutils import hhmm_to_minutes
from xora.plotting import apply_style, SERIES

apply_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. File inventory

The data zip has four folders. Here is every CSV with its size and shape.

In [2]:
inventory = []
for path in sorted(DATA_RAW.rglob("*.csv")):
    df = pd.read_csv(path)
    inventory.append({
        "folder": path.parent.name,
        "file": path.name,
        "rows": len(df),
        "columns": df.shape[1],
    })
pd.DataFrame(inventory)

,folder,file,rows,columns
0,General Data,calendar.csv,910,12
1,General Data,district_travel.csv,12,8
2,General Data,outlets.csv,120,9
3,General Data,road_conditions.csv,10920,3
4,General Data,service_allowance.csv,9,3
5,General Data,traffic_speed.csv,576,4
6,General Data,vehicles.csv,60,9
7,Submission Templates,submission_task1.csv,5014,3
8,Submission Templates,submission_task2a.csv,60,3
9,Submission Templates,submission_task2b.csv,85,6


In [3]:
orders = load_raw("deliveries_train.csv")
legs = load_raw("route_legs_train.csv")
orders_test = load_raw("task1_test_inputs.csv")
legs_test = load_raw("route_legs_test.csv")
outlets = load_raw("outlets.csv")
vehicles = load_raw("vehicles.csv")
calendar = load_raw("calendar.csv")
district_travel = load_raw("district_travel.csv")
allowance = load_raw("service_allowance.csv")
road = load_raw("road_conditions.csv")
traffic = load_raw("traffic_speed.csv")

**How the files connect**

- `deliveries_train.csv` has one row per order. Each dispatched order points to a route (`route_id`) and its position on that route (`seq_in_route`).
- `route_legs_train.csv` has one row per leg: a drive from the depot or the previous stop to the next outlet. It holds the actual times we need for labels.
- `outlets.csv`, `vehicles.csv`, `district_travel.csv` and `service_allowance.csv` describe the network and the current planning standards.
- `calendar.csv`, `road_conditions.csv` and `traffic_speed.csv` describe each day: paydays, festivals, monsoon, road disruption and traffic speed by hour.
- The test files mirror the train files but only carry planned times.